In [1]:
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report, confusion_matrix

In [2]:
train_df = pd.read_csv("../data/train_processed.csv")
test_df = pd.read_csv("../data/test_processed.csv")

In [3]:
features = [
    'T2M', 'RH2M', 'PS',
    'hour', 'month',
    'T2M_diff', 'RH2M_diff', 'PS_diff',
    'T2M_roll_mean', 'RH2M_roll_mean', 'PS_roll_mean',
    'T2M_roll_std', 'RH2M_roll_std', 'PS_roll_std',
    'T2M_roll_dev', 'RH2M_roll_dev', 'PS_roll_dev'
]

X_train = train_df[features]

X_test = test_df[features]
y_test = test_df['is_anomaly']

In [9]:
model = IsolationForest(
    contamination=0.03,
    random_state=42
)

model.fit(X_train)

pred = model.predict(X_test)
y_pred = (pred == -1).astype(int)

results = test_df.copy()
results['predicted_anomaly'] = y_pred

for anomaly in results['anomaly_type'].unique():
    subset = results[results['anomaly_type'] == anomaly]

    if anomaly != 'normal':
        detected = subset['predicted_anomaly'].mean()

        print(
            anomaly,
            "->",
            round(detected * 100, 2),
            "% detected"
        )

temperature_spike -> 90.16 % detected
multivariate_inconsistency -> 85.33 % detected
temperature_frozen -> 17.71 % detected
temperature_drift -> 16.13 % detected


In [11]:
test_df['T2M_frozen'] = (
    test_df['T2M'].rolling(6).std() < 0.05
)

In [12]:
frozen_rows = test_df[
    test_df['anomaly_type'] == 'temperature_frozen'
]

print(
    frozen_rows['T2M_frozen'].mean() * 100,
    "% frozen anomalies detected"
)

55.208333333333336 % frozen anomalies detected
